### 做数据处理过，并且已经跑了一遍模型后，再做的特征工程
- 计算每个数值特征与`SeriousDlqin2yrs`的皮尔逊相关系数，观察绝对值大小。而皮尔逊相关系数只能捕捉线性关系，此时可能显示为0。
- 查看模型的特征重要性（树模型）或系数（线性模型），找出贡献低的特征，考虑是否删除或变换。
- 做相关性分析，了解基本关系，构造一些直观的简单特征（如总逾期次数、收入/负债比）。

In [1]:
import os
import pandas as pd

# 1) 自动定位处理后的训练集（优先 extra，其次 new）
candidate_paths = [
    "../data/processed/train_set_processed_new.csv",
]

train_path = None
for path in candidate_paths:
    if os.path.exists(path):
        train_path = os.path.abspath(path)
        break

if train_path is None:
    raise FileNotFoundError("未找到可用训练集，请检查 data/processed 路径。")

# 2) 读取数据（当前流程统一为无索引导出）
df = pd.read_csv(train_path)

# 兼容历史文件：若存在历史索引列（Unnamed:*），自动移除
unnamed_cols = [c for c in df.columns if str(c).startswith("Unnamed:")]
if unnamed_cols:
    df = df.drop(columns=unnamed_cols)
    print(f"检测到并移除历史索引列: {unnamed_cols}")

target_col = "SeriousDlqin2yrs"

if target_col not in df.columns:
    raise ValueError(f"未找到目标列: {target_col}")

# 3) 仅保留数值列并计算皮尔逊相关系数
numeric_df = df.select_dtypes(include=["number"])

if target_col not in numeric_df.columns:
    raise ValueError(f"目标列 {target_col} 不是数值列，无法直接计算皮尔逊相关。")

pearson_corr = numeric_df.corr(method="pearson")[target_col].drop(labels=[target_col])
result_df = (
    pearson_corr
    .to_frame(name="pearson_corr")
    .assign(abs_corr=lambda x: x["pearson_corr"].abs())
    .sort_values("abs_corr", ascending=False)
)

# 4) 控制台输出
print(f"当前工作目录: {os.getcwd()}")
print(f"数据文件: {train_path}")
print(f"样本数: {len(df)}")
print(f"数值特征数: {result_df.shape[0]}")
print("\n各数值特征与 SeriousDlqin2yrs 的皮尔逊相关系数（按绝对值降序）：")
print(result_df.to_string(float_format=lambda x: f"{x:.6f}"))

当前工作目录: f:\graduation-project\Credit-card-delinquency-prediction-based-on-DNN\notebooks
数据文件: f:\graduation-project\Credit-card-delinquency-prediction-based-on-DNN\data\processed\train_set_processed_new.csv
样本数: 116306
数值特征数: 10

各数值特征与 SeriousDlqin2yrs 的皮尔逊相关系数（按绝对值降序）：
                                      pearson_corr  abs_corr
NumberOfTimes90DaysLate                   0.315797  0.315797
NumberOfTime30-59DaysPastDueNotWorse      0.277337  0.277337
NumberOfTime60-89DaysPastDueNotWorse      0.267375  0.267375
age                                      -0.111408  0.111408
NumberOfDependents                        0.046324  0.046324
MonthlyIncome                            -0.027364  0.027364
NumberOfOpenCreditLinesAndLoans          -0.026529  0.026529
DebtRatio                                -0.007315  0.007315
NumberRealEstateLoansOrLines             -0.006353  0.006353
RevolvingUtilizationOfUnsecuredLines     -0.000309  0.000309


### 计算三个逾期特征之间的相关系数

In [5]:
# 计算三个逾期特征之间的皮尔逊相关系数
# 复用第 2 个单元已读取的数据，避免重复读取文件
if "df" not in globals():
    raise NameError("未找到变量 df，请先运行第 2 个单元（读取 train_set.csv）。")

# Give Me Some Credit 数据集中常用的三个逾期特征
overdue_features = [
    "NumberOfTime30-59DaysPastDueNotWorse",
    "NumberOfTime60-89DaysPastDueNotWorse",
    "NumberOfTimes90DaysLate",
]

missing_cols = [col for col in overdue_features if col not in df.columns]
if missing_cols:
    raise ValueError(f"以下逾期特征不存在: {missing_cols}")

corr_matrix = df[overdue_features].corr(method="pearson")

print("三个逾期特征的皮尔逊相关系数矩阵：")
print(corr_matrix.to_string(float_format=lambda x: f"{x:.6f}"))

三个逾期特征的皮尔逊相关系数矩阵：
                                      NumberOfTime30-59DaysPastDueNotWorse  NumberOfTime60-89DaysPastDueNotWorse  NumberOfTimes90DaysLate
NumberOfTime30-59DaysPastDueNotWorse                              1.000000                              0.309506                 0.220054
NumberOfTime60-89DaysPastDueNotWorse                              0.309506                              1.000000                 0.291190
NumberOfTimes90DaysLate                                           0.220054                              0.291190                 1.000000


### 新特征之间的关联性

In [ ]:
feature_aliases = {
    "Utilization_x_TotalLate": ["Utilization_x_TotalLate", "Utilization_x_Totallate"],
    "TotalLateTimes": ["TotalLateTimes"],
    "RevolvingUtilizationOfUnsecuredLines": ["RevolvingUtilizationOfUnsecuredLines", "RevolvingUtilization"],
    "HighUtil_And_Late": ["HighUtil_And_Late", "Highutil_And_Late"],
    "IsUtilMaxed": ["IsUtilMaxed"],
    "EstMonthlyDebt": ["EstMonthlyDebt"],
    "IncomePerDependent": ["IncomePerDependent"],
}

selected_cols = []
missing_features = []

for display_name, candidates in feature_aliases.items():
    matched_col = next((col for col in candidates if col in df.columns), None)
    if matched_col is None:
        missing_features.append(display_name)
    else:
        selected_cols.append(matched_col)

if missing_features:
    print(f"以下候选特征在 df 中不存在（已跳过）: {missing_features}")

if len(selected_cols) < 2:
    raise ValueError(
        f"可用于相关性分析的特征不足 2 列。当前命中列: {selected_cols}。"
    )

corr_matrix = df[selected_cols].corr()
print("Selected columns:", selected_cols)
print(corr_matrix)
corr_matrix

Selected columns: ['Utilization_x_TotalLate', 'TotalLateTimes', 'RevolvingUtilizationOfUnsecuredLines', 'HighUtil_And_Late', 'IsUtilMaxed']
                                      Utilization_x_TotalLate  TotalLateTimes  \
Utilization_x_TotalLate                              1.000000        0.894512   
TotalLateTimes                                       0.894512        1.000000   
RevolvingUtilizationOfUnsecuredLines                 0.424101        0.319349   
HighUtil_And_Late                                    0.692119        0.545809   
IsUtilMaxed                                          0.311122        0.231367   

                                      RevolvingUtilizationOfUnsecuredLines  \
Utilization_x_TotalLate                                           0.424101   
TotalLateTimes                                                    0.319349   
RevolvingUtilizationOfUnsecuredLines                              1.000000   
HighUtil_And_Late                                            

,Utilization_x_TotalLate,TotalLateTimes,RevolvingUtilizationOfUnsecuredLines,HighUtil_And_Late,IsUtilMaxed
Utilization_x_TotalLate,1.000000,0.894512,0.424101,0.692119,0.311122
TotalLateTimes,0.894512,1.000000,0.319349,0.545809,0.231367
RevolvingUtilizationOfUnsecuredLines,0.424101,0.319349,1.000000,0.519473,0.294889
HighUtil_And_Late,0.692119,0.545809,0.519473,1.000000,0.334690
IsUtilMaxed,0.311122,0.231367,0.294889,0.334690,1.000000
